# SPY Seasonal Edge Analysis

**Analyzes SPY performance across VIX Z-Score regimes**

## Market Seasons (Regimes)

- **Summer** (Green): RiskZ > 0 and rising → Risk-On accelerating
- **Fall** (Dark Green): RiskZ > 0 but falling → Risk-On weakening
- **Winter** (Red): RiskZ ≤ 0 and falling → Risk-Off accelerating
- **Spring** (Dark Red): RiskZ ≤ 0 but rising → Risk-Off weakening

## Edges Analyzed

1. **Overnight Returns** (Close-to-Open) by season
2. **Intraday Returns** (Open-to-Close) by season
3. **Swing Period Performance** (2D, 3D, 5D, 10D holds) by season
4. **Statistical edges** and optimal holding periods

## Install Dependencies

In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn

## Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')

## Configuration

In [ ]:
START_DATE = '2010-01-01'
END_DATE = datetime.today().strftime('%Y-%m-%d')

# Risk Z parameters (matching ThinkScript)
VIX_FAST = 10
VIX_SLOW = 30
Z_LOOKBACK = 180

print(f"Analysis Period: {START_DATE} to {END_DATE}")

## Download Data

In [ ]:
print("Downloading SPY and VIX data...")

# Download SPY with full OHLC
spy = yf.download('SPY', start=START_DATE, end=END_DATE, progress=True)

# Download VIX
vix = yf.download('^VIX', start=START_DATE, end=END_DATE, progress=True)

# Combine data
data = pd.DataFrame()
data['SPY_Open'] = spy['Open']
data['SPY_High'] = spy['High']
data['SPY_Low'] = spy['Low']
data['SPY_Close'] = spy['Close']
data['SPY_Volume'] = spy['Volume']
data['VIX_Close'] = vix['Close']

data = data.dropna()

print(f"\nDownloaded {len(data)} trading days")
data.head()

## Calculate Risk Z-Score

In [ ]:
print("Calculating Risk Z-Score...")

# VIX moving averages
data['VIX_Fast'] = data['VIX_Close'].rolling(window=VIX_FAST).mean()
data['VIX_Slow'] = data['VIX_Close'].rolling(window=VIX_SLOW).mean()

# Risk Index (VIX_Slow / VIX_Fast)
data['Risk_Index'] = data['VIX_Slow'] / data['VIX_Fast']

# Z-Score normalization
risk_mean = data['Risk_Index'].rolling(window=Z_LOOKBACK).mean()
risk_std = data['Risk_Index'].rolling(window=Z_LOOKBACK).std()

data['Risk_Z'] = (data['Risk_Index'] - risk_mean) / risk_std

data = data.dropna()

print(f"Risk Z calculated. {len(data)} valid data points.")

# Show Risk Z statistics
print("\nRisk Z Statistics:")
print(data['Risk_Z'].describe())

## Classify Seasons

In [ ]:
print("Classifying seasons...")

# Calculate if Risk Z is rising or falling
data['Risk_Z_Change'] = data['Risk_Z'].diff()

# Season classification
conditions = [
    (data['Risk_Z'] > 0) & (data['Risk_Z_Change'] >= 0),  # Summer
    (data['Risk_Z'] > 0) & (data['Risk_Z_Change'] < 0),   # Fall
    (data['Risk_Z'] <= 0) & (data['Risk_Z_Change'] <= 0), # Winter
    (data['Risk_Z'] <= 0) & (data['Risk_Z_Change'] > 0)   # Spring
]

choices = ['Summer', 'Fall', 'Winter', 'Spring']

data['Season'] = np.select(conditions, choices, default='Unknown')

# Season distribution
season_counts = data['Season'].value_counts()
print("\nSeason Distribution:")
for season, count in season_counts.items():
    pct = count / len(data) * 100
    print(f"  {season}: {count} days ({pct:.1f}%)")

# Visualize season distribution
season_counts.plot(kind='bar', color=['green', 'orange', 'red', 'lightcoral'])
plt.title('Season Distribution')
plt.xlabel('Season')
plt.ylabel('Number of Days')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

## Calculate Returns

In [ ]:
print("Calculating returns...")

# Overnight return (prev close to current open)
data['Overnight_Return'] = (
    (data['SPY_Open'] - data['SPY_Close'].shift(1)) /
    data['SPY_Close'].shift(1) * 100
)

# Intraday return (open to close)
data['Intraday_Return'] = (
    (data['SPY_Close'] - data['SPY_Open']) /
    data['SPY_Open'] * 100
)

# Daily return (close to close)
data['Daily_Return'] = data['SPY_Close'].pct_change() * 100

# Swing period returns
data['Swing_2D'] = (
    (data['SPY_Close'].shift(-2) - data['SPY_Close']) /
    data['SPY_Close'] * 100
)

data['Swing_3D'] = (
    (data['SPY_Close'].shift(-3) - data['SPY_Close']) /
    data['SPY_Close'] * 100
)

data['Swing_5D'] = (
    (data['SPY_Close'].shift(-5) - data['SPY_Close']) /
    data['SPY_Close'] * 100
)

data['Swing_10D'] = (
    (data['SPY_Close'].shift(-10) - data['SPY_Close']) /
    data['SPY_Close'] * 100
)

print("Returns calculated!")

# Show sample data
data[['Season', 'Risk_Z', 'Overnight_Return', 'Intraday_Return', 'Daily_Return']].tail(10)

## Analyze by Season

In [ ]:
print("="*80)
print("SEASONAL EDGE ANALYSIS")
print("="*80)

seasons = ['Summer', 'Fall', 'Winter', 'Spring']
results = []

for season in seasons:
    season_data = data[data['Season'] == season].copy()
    
    if len(season_data) == 0:
        continue
    
    stats = {
        'Season': season,
        'Days': len(season_data),
        
        # Overnight stats
        'Overnight_Mean': season_data['Overnight_Return'].mean(),
        'Overnight_Median': season_data['Overnight_Return'].median(),
        'Overnight_Std': season_data['Overnight_Return'].std(),
        'Overnight_Win%': (season_data['Overnight_Return'] > 0).sum() / len(season_data) * 100,
        'Overnight_Sharpe': (season_data['Overnight_Return'].mean() /
                            season_data['Overnight_Return'].std() * np.sqrt(252)),
        
        # Intraday stats
        'Intraday_Mean': season_data['Intraday_Return'].mean(),
        'Intraday_Median': season_data['Intraday_Return'].median(),
        'Intraday_Std': season_data['Intraday_Return'].std(),
        'Intraday_Win%': (season_data['Intraday_Return'] > 0).sum() / len(season_data) * 100,
        'Intraday_Sharpe': (season_data['Intraday_Return'].mean() /
                           season_data['Intraday_Return'].std() * np.sqrt(252)),
        
        # Daily stats
        'Daily_Mean': season_data['Daily_Return'].mean(),
        'Daily_Std': season_data['Daily_Return'].std(),
        'Daily_Win%': (season_data['Daily_Return'] > 0).sum() / len(season_data) * 100,
        
        # Swing stats
        'Swing_2D_Mean': season_data['Swing_2D'].mean(),
        'Swing_3D_Mean': season_data['Swing_3D'].mean(),
        'Swing_5D_Mean': season_data['Swing_5D'].mean(),
        'Swing_10D_Mean': season_data['Swing_10D'].mean(),
    }
    
    results.append(stats)

analysis = pd.DataFrame(results)

# Print results
for idx, row in analysis.iterrows():
    season = row['Season']
    print(f"\n{'='*80}")
    print(f"{season.upper()} (Risk-{'On' if season in ['Summer', 'Fall'] else 'Off'})")
    print(f"{'='*80}")
    print(f"Total Days: {row['Days']:.0f}")
    
    print(f"\n--- OVERNIGHT (Close-to-Open) ---")
    print(f"  Mean Return:    {row['Overnight_Mean']:>8.3f}%")
    print(f"  Median Return:  {row['Overnight_Median']:>8.3f}%")
    print(f"  Std Dev:        {row['Overnight_Std']:>8.3f}%")
    print(f"  Win Rate:       {row['Overnight_Win%']:>8.1f}%")
    print(f"  Sharpe Ratio:   {row['Overnight_Sharpe']:>8.2f}")
    
    print(f"\n--- INTRADAY (Open-to-Close) ---")
    print(f"  Mean Return:    {row['Intraday_Mean']:>8.3f}%")
    print(f"  Median Return:  {row['Intraday_Median']:>8.3f}%")
    print(f"  Std Dev:        {row['Intraday_Std']:>8.3f}%")
    print(f"  Win Rate:       {row['Intraday_Win%']:>8.1f}%")
    print(f"  Sharpe Ratio:   {row['Intraday_Sharpe']:>8.2f}")
    
    print(f"\n--- DAILY (Close-to-Close) ---")
    print(f"  Mean Return:    {row['Daily_Mean']:>8.3f}%")
    print(f"  Std Dev:        {row['Daily_Std']:>8.3f}%")
    print(f"  Win Rate:       {row['Daily_Win%']:>8.1f}%")
    
    print(f"\n--- SWING PERIODS ---")
    print(f"  2-Day Mean:     {row['Swing_2D_Mean']:>8.3f}%")
    print(f"  3-Day Mean:     {row['Swing_3D_Mean']:>8.3f}%")
    print(f"  5-Day Mean:     {row['Swing_5D_Mean']:>8.3f}%")
    print(f"  10-Day Mean:    {row['Swing_10D_Mean']:>8.3f}%")

## Edge Summary

In [ ]:
print("="*80)
print("EDGE SUMMARY")
print("="*80)

# Find best season for each metric
best_overnight = analysis.loc[analysis['Overnight_Mean'].idxmax()]
best_intraday = analysis.loc[analysis['Intraday_Mean'].idxmax()]
best_daily = analysis.loc[analysis['Daily_Mean'].idxmax()]

print(f"\nBest Overnight Edge:  {best_overnight['Season']} ({best_overnight['Overnight_Mean']:.3f}%/day)")
print(f"Best Intraday Edge:   {best_intraday['Season']} ({best_intraday['Intraday_Mean']:.3f}%/day)")
print(f"Best Daily Edge:      {best_daily['Season']} ({best_daily['Daily_Mean']:.3f}%/day)")

# Overnight vs Intraday comparison
print(f"\n--- Overnight vs Intraday by Season ---")
for idx, row in analysis.iterrows():
    on_edge = row['Overnight_Mean']
    id_edge = row['Intraday_Mean']
    better = "OVERNIGHT" if on_edge > id_edge else "INTRADAY"
    diff = abs(on_edge - id_edge)
    print(f"{row['Season']:10s}: {better:10s} edge (+{diff:.3f}%)")

## Visualizations

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))

seasons = ['Summer', 'Fall', 'Winter', 'Spring']
colors_map = {'Summer': 'green', 'Fall': 'orange', 'Winter': 'red', 'Spring': 'lightcoral'}

# Plot 1: Overnight vs Intraday Returns
ax = axes[0, 0]
overnight = [analysis[analysis['Season']==s]['Overnight_Mean'].values[0] for s in seasons]
intraday = [analysis[analysis['Season']==s]['Intraday_Mean'].values[0] for s in seasons]

x = np.arange(len(seasons))
width = 0.35

ax.bar(x - width/2, overnight, width, label='Overnight', alpha=0.8)
ax.bar(x + width/2, intraday, width, label='Intraday', alpha=0.8)
ax.set_xlabel('Season')
ax.set_ylabel('Mean Return (%)')
ax.set_title('Overnight vs Intraday Returns by Season', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(seasons)
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Plot 2: Win Rates
ax = axes[0, 1]
overnight_wr = [analysis[analysis['Season']==s]['Overnight_Win%'].values[0] for s in seasons]
intraday_wr = [analysis[analysis['Season']==s]['Intraday_Win%'].values[0] for s in seasons]

ax.bar(x - width/2, overnight_wr, width, label='Overnight', alpha=0.8)
ax.bar(x + width/2, intraday_wr, width, label='Intraday', alpha=0.8)
ax.set_xlabel('Season')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rates by Season', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(seasons)
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=50, color='black', linestyle='--', linewidth=0.5, alpha=0.5)

# Plot 3: Sharpe Ratios
ax = axes[1, 0]
overnight_sharpe = [analysis[analysis['Season']==s]['Overnight_Sharpe'].values[0] for s in seasons]
intraday_sharpe = [analysis[analysis['Season']==s]['Intraday_Sharpe'].values[0] for s in seasons]

ax.bar(x - width/2, overnight_sharpe, width, label='Overnight', alpha=0.8)
ax.bar(x + width/2, intraday_sharpe, width, label='Intraday', alpha=0.8)
ax.set_xlabel('Season')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Risk-Adjusted Returns (Sharpe Ratio)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(seasons)
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Plot 4: Swing Period Returns
ax = axes[1, 1]
swing_periods = ['2D', '3D', '5D', '10D']
for season in seasons:
    row = analysis[analysis['Season'] == season].iloc[0]
    swing_returns = [row['Swing_2D_Mean'], row['Swing_3D_Mean'],
                   row['Swing_5D_Mean'], row['Swing_10D_Mean']]
    ax.plot(swing_periods, swing_returns, marker='o', label=season,
           color=colors_map[season], linewidth=2)

ax.set_xlabel('Swing Period')
ax.set_ylabel('Mean Return (%)')
ax.set_title('Swing Period Returns by Season', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Plot 5: Return Distributions (Overnight)
ax = axes[2, 0]
for season in seasons:
    season_data = data[data['Season'] == season]['Overnight_Return'].dropna()
    ax.hist(season_data, bins=50, alpha=0.5, label=season, color=colors_map[season])

ax.set_xlabel('Overnight Return (%)')
ax.set_ylabel('Frequency')
ax.set_title('Overnight Return Distributions', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Plot 6: Return Distributions (Intraday)
ax = axes[2, 1]
for season in seasons:
    season_data = data[data['Season'] == season]['Intraday_Return'].dropna()
    ax.hist(season_data, bins=50, alpha=0.5, label=season, color=colors_map[season])

ax.set_xlabel('Intraday Return (%)')
ax.set_ylabel('Frequency')
ax.set_title('Intraday Return Distributions', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## Export to Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Export analysis summary
analysis.to_csv('/content/drive/MyDrive/seasonal_analysis.csv', index=False)
print("Analysis exported to Google Drive: seasonal_analysis.csv")

# Export detailed day-by-day data
export_cols = ['Season', 'Risk_Z', 'SPY_Close', 'Overnight_Return',
              'Intraday_Return', 'Daily_Return', 'Swing_2D',
              'Swing_3D', 'Swing_5D', 'Swing_10D']
data[export_cols].to_csv('/content/drive/MyDrive/seasonal_detail.csv')
print("Detailed data exported to Google Drive: seasonal_detail.csv")

# Save the chart
fig.savefig('/content/drive/MyDrive/seasonal_edge_analysis.png', dpi=300, bbox_inches='tight')
print("Chart saved to Google Drive: seasonal_edge_analysis.png")

## Current Market Status

In [ ]:
# Show current season and expected edges
current_season = data['Season'].iloc[-1]
current_z = data['Risk_Z'].iloc[-1]
current_spy = data['SPY_Close'].iloc[-1]

print(f"\n{'='*80}")
print("CURRENT MARKET STATUS")
print(f"{'='*80}")
print(f"Current Season: {current_season}")
print(f"Risk Z-Score:   {current_z:.2f}")
print(f"SPY Price:      ${current_spy:.2f}")

# Show expected edges for current season
current_stats = analysis[analysis['Season'] == current_season].iloc[0]
print(f"\nExpected Edges for {current_season}:")
print(f"  Overnight Mean:  {current_stats['Overnight_Mean']:.3f}%")
print(f"  Intraday Mean:   {current_stats['Intraday_Mean']:.3f}%")
print(f"  Daily Mean:      {current_stats['Daily_Mean']:.3f}%")
print(f"  5-Day Swing:     {current_stats['Swing_5D_Mean']:.3f}%")

better_edge = "OVERNIGHT" if current_stats['Overnight_Mean'] > current_stats['Intraday_Mean'] else "INTRADAY"
print(f"\nRecommended: Trade {better_edge} edge in {current_season} regime")